In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_selector as selector
from sklearn.impute import SimpleImputer

# =============================================================================
def split_dataset(
    dataset,
    label_col="Cancer_Type",
    test_size=0.15,
    val_size=0.15,
    random_state=42,
):
    """
    Stratified Train/Validation/Test split.
    """

    train_df, temp_df = train_test_split(
        dataset,
        test_size=test_size + val_size,
        stratify=dataset[label_col],
        random_state=random_state,
    )

    relative_val = val_size / (test_size + val_size)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=1 - relative_val,
        stratify=temp_df[label_col],
        random_state=random_state,
    )

    return (
         train_df.copy(),
         val_df.copy(),
         test_df.copy(),
    )


# =============================================================================
def preprocess(dataset):

    print("=" * 60)
    print("Splitting Dataset")
    print("=" * 60)

    train_df, val_df, test_df = split_dataset(dataset)

    print("Train:", train_df.shape)
    print("Validation:", val_df.shape)
    print("Test:", test_df.shape)


    

    print()

    print("=" * 60)
    print("Learning Imputation Statistics")
    print("=" * 60)

    # Numerical columns (gene expression)
    numerical_columns_selector = selector(dtype_include=["number"])

    # Categorical columns (Sample_Type, Cancer_Type)
    categorical_columns_selector = selector(dtype_exclude=["number"])

    numerical_columns = numerical_columns_selector(train_df)
    categorical_columns = categorical_columns_selector(train_df)

    print("Number of numerical columns:", len(numerical_columns))
    print("Categorical columns:", categorical_columns)
    #print("numerical columns:", numerical_columns)

    # Numerical (genes)
    gene_imputer = SimpleImputer(strategy="mean")

    train_df[numerical_columns] = gene_imputer.fit_transform(
    train_df[numerical_columns])

    val_df[numerical_columns] = gene_imputer.transform(
    val_df[numerical_columns])

    test_df[numerical_columns] = gene_imputer.transform(
    test_df[numerical_columns])

    os.makedirs("./models", exist_ok=True)
    joblib.dump(gene_imputer, "./models/gene_imputer.pkl")


    print()

    print("=" * 60)
    print("Missing Values After Imputation")
    print("=" * 60)

    print("Train:", train_df.isnull().sum().sum())
    print("Validation:", val_df.isnull().sum().sum())
    print("Test:", test_df.isnull().sum().sum())

    return train_df, val_df, test_df

# =============================================================================
def save_processed_data(
    train_df,
    val_df,
    test_df,
    output_dir="./data/preprocessed_data",
):

    

    os.makedirs(output_dir, exist_ok=True)

    train_df.to_csv(
        os.path.join(output_dir, "train.csv"),
        index=False,
    )

    val_df.to_csv(
        os.path.join(output_dir, "validation.csv"),
        index=False,
    )

    test_df.to_csv(
        os.path.join(output_dir, "test.csv"),
        index=False,
    )

    print()

    print("=" * 60)
    print("Datasets Saved")
    print("=" * 60)

    print(output_dir)

# =============================================================================
#MAIN 
dataset = pd.read_csv("TCGA_Cancer_Dataset.csv",index_col=0)
print(np.shape(dataset))
train_df, val_df, test_df = preprocess(dataset)



print("\nTrain cancer distribution:")
print(np.shape(train_df))
print(train_df['Cancer_Type'].value_counts())

print("\nValidation cancer distribution:")
print(np.shape(val_df))
print(val_df['Cancer_Type'].value_counts())

print("\nTest cancer distribution:")
print(np.shape(test_df))
print(test_df['Cancer_Type'].value_counts())
save_processed_data(train_df, val_df, test_df)

Train_data =pd.read_csv('F:\\GeneformerProject\\data\\preprocessed_data\\train.csv',index_col=0)
print("check the train size=",np.shape(Train_data))
Test_data =pd.read_csv('F:\\GeneformerProject\\data\\preprocessed_data\\test.csv')
print("check the test size:", np.shape(Test_data))

(11060, 20533)
Splitting Dataset
Train: (7742, 20533)
Validation: (1659, 20533)
Test: (1659, 20533)

Learning Imputation Statistics
Number of numerical columns: 20531
Categorical columns: ['Cancer_Type', 'tissue_type']

Missing Values After Imputation
Train: 0
Validation: 0
Test: 0

Train cancer distribution:
(7742, 20533)
Cancer_Type
breast invasive carcinoma                853
kidney clear cell carcinoma              424
lung adenocarcinoma                      403
thyroid carcinoma                        400
uterine corpus endometrioid carcinoma    397
head & neck squamous cell carcinoma      396
lung squamous cell carcinoma             387
prostate adenocarcinoma                  385
brain lower grade glioma                 371
colon adenocarcinoma                     347
skin cutaneous melanoma                  332
stomach adenocarcinoma                   315
bladder urothelial carcinoma             299
liver hepatocellular carcinoma           296
kidney papillary cell carcinoma  